# PLC Data: Initial Quality Check (Siemens S7)

**Goal:** Load the raw PLC recordings, find data-quality problems and
clean them, so the data is ready for later steps such as resampling,
feature engineering and stop detection.

**Data**
- `Beispieldatensatz Siemens S7 SPS.csv` - the dataset checked in this
  notebook.
- `Beispieldatensatz Beckhoff SPS.csv` - loaded here, but not analysed
  yet.

**Rule:** the raw data is never changed. All cleaning is done on a copy.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

SIEMENS_PATH = RAW_DATA_DIR / "Beispieldatensatz Siemens S7 SPS.csv"
BECKHOFF_PATH = RAW_DATA_DIR / "Beispieldatensatz Beckhoff SPS.csv"

## 2. Load the data

The files use `;` as separator.

`low_memory=False` makes pandas read each column in one go. Without it,
pandas printed a `DtypeWarning` about mixed types in more than 100
columns. The cause of those mixed types is explained in section 3.

In [2]:
siemens_data = pd.read_csv(SIEMENS_PATH, sep=";", low_memory=False)

print("PLC datasets loaded successfully.")

PLC datasets loaded successfully.


## 3. First look at the raw Siemens data

In [3]:
print(f"Rows: {siemens_data.shape[0]:,}")
print(f"Columns: {siemens_data.shape[1]}")
print(f"Duplicate rows: {siemens_data.duplicated().sum():,}")

missing_per_column = siemens_data.isna().sum()
missing_per_column = missing_per_column[missing_per_column > 0]
print(f"Columns with missing values: {len(missing_per_column)}")
print(f"Rows with missing values: {siemens_data.isna().any(axis=1).sum()}")

Rows: 6,283
Columns: 144
Duplicate rows: 0
Columns with missing values: 119
Rows with missing values: 6283


## 4. Cleaning

### 4.1 Remove the empty column and the incomplete row

The incomplete row has 118 missing values out of 143 columns, so only
25 values are left. Keeping it would add a half-empty observation to
later steps (signal processing, feature engineering, stop detection).
Removing one row out of 6,283 costs almost nothing.

This decision is based on the missing-value pattern only. It does not
assume that the machine was stopped.

In [4]:
siemens_clean = siemens_data.copy()

# Drop columns that are empty everywhere.
empty_columns = siemens_clean.columns[siemens_clean.isna().all()]
siemens_clean = siemens_clean.drop(columns=empty_columns)
print("Dropped empty columns:", empty_columns.tolist())


# Drop rows where more than half of the values are missing.
mostly_missing = siemens_clean.isna().sum(axis=1) > siemens_clean.shape[1] / 2
print("Dropped row index:", siemens_clean.index[mostly_missing].tolist())
siemens_clean = siemens_clean[~mostly_missing]

print("\nRaw shape:    ", siemens_data.shape)
print("Cleaned shape:", siemens_clean.shape)
print("Rows with missing values:", siemens_clean.isna().any(axis=1).sum())

Dropped empty columns: ['Unnamed: 143']
Dropped row index: [6282]

Raw shape:     (6283, 144)
Cleaned shape: (6282, 143)
Rows with missing values: 0


### 4.2 Convert `object` columns to `bool`

Before the row was removed, 103 columns had the type `object`. We check
that they only contain `True`/`False` and then convert them.

In [5]:
object_columns = siemens_clean.select_dtypes(include="object").columns

all_boolean = all(
    siemens_clean[column].map(lambda value: isinstance(value, bool)).all()
    for column in object_columns
)
print("Number of object columns:", len(object_columns))
print("All values are Boolean:", all_boolean)

if all_boolean:
    siemens_clean[object_columns] = siemens_clean[object_columns].astype(bool)

print("\nData types after conversion:")
print(siemens_clean.dtypes.value_counts())

Number of object columns: 103
All values are Boolean: True

Data types after conversion:
bool       125
float64     15
int64        3
Name: count, dtype: int64


## 5. Constant columns

A constant column has the same value for the whole recording. Such a
signal carries no information **in this recording**. It still might be
needed later (for example if it is used in the feature-engineering
code), so we only list them here and do not remove them yet.

In [6]:
unique_counts = siemens_clean.nunique()

constant_columns = unique_counts[unique_counts <= 1].index.tolist()
variable_columns = unique_counts[unique_counts > 1].index.tolist()

print("Total columns:   ", len(siemens_clean.columns))
print("Constant columns:", len(constant_columns))
print("Variable columns:", len(variable_columns))

constant_summary = pd.DataFrame({
    "data_type": siemens_clean[constant_columns].dtypes.astype(str),
    "constant_value": siemens_clean[constant_columns].iloc[0],
})
print()
print(constant_summary.to_string())

Total columns:    143
Constant columns: 32
Variable columns: 111

       data_type constant_value
M34.4       bool          False
M38.0       bool          False
M38.1       bool          False
C0       float64            0.0
EW288    float64            0.0
C1       float64            0.0
C2       float64            0.0
M2.0        bool          False
M2.1        bool          False
M2.2        bool          False
M2.3        bool          False
M3.0        bool          False
M3.1        bool          False
C4       float64            0.0
M4.0        bool          False
M5.7        bool          False
M7.0        bool          False
A6.4        bool          False
A6.5        bool          False
M24.0       bool          False
M24.1       bool          False
A201.1      bool           True
M24.2       bool          False
EW258      int64              1
EW256      int64              1
M24.5       bool          False
M0.0        bool           True
E6.0        bool           True
E6.1  

In [7]:
multi_valued = unique_counts[unique_counts > 2].sort_values()
print(multi_valued.to_string())

MW12            3
MW20            3
MW22            4
AW256           5
MW26            7
MW18            8
MW14           13
MW16           46
timestamp    6282


## 6. Timestamps

The `timestamp` column is a Unix time in milliseconds. We check how
regular the sampling is.

In [8]:
timestamps = siemens_clean["timestamp"]
intervals = timestamps.diff().dropna()

print("Median interval (ms): ", intervals.median())
print("Maximum interval (ms):", intervals.max())

for threshold in [100, 500, 1000]:
    print(f"Intervals above {threshold} ms:", (intervals > threshold).sum())

Median interval (ms):  76.0
Maximum interval (ms): 1722.0
Intervals above 100 ms: 124
Intervals above 500 ms: 12
Intervals above 1000 ms: 4


In [9]:
gaps = pd.DataFrame({
    "previous_timestamp": timestamps.shift().astype("Int64"),
    "current_timestamp": timestamps,
    "gap_ms": timestamps.diff(),
})
long_gaps = gaps[gaps["gap_ms"] > 1000]

print("Gaps longer than 1 second:", len(long_gaps))
print(long_gaps.to_string())

Gaps longer than 1 second: 4
      previous_timestamp  current_timestamp  gap_ms
376        1569522104557      1569522106279  1722.0
1554       1569522198415      1569522199429  1014.0
2839       1569522304406      1569522305585  1179.0
3869       1569522388392      1569522389758  1366.0


## 7. Final checks

In [10]:
print("Dataset shape:", siemens_clean.shape)
print("Total missing values:", siemens_clean.isna().sum().sum())
print("Timestamps increasing:", timestamps.is_monotonic_increasing)
print("Duplicate timestamps:", timestamps.duplicated().sum())

Dataset shape: (6282, 143)
Total missing values: 0
Timestamps increasing: True
Duplicate timestamps: 0


## 8. Summary

**Cleaning done**
- Removed the empty column `Unnamed: 143`.
- Removed one incomplete row (original index 6282, 118 of 143 values
  missing).
- Converted 103 `object` columns to `bool`.
- Kept the raw data unchanged in `siemens_data`.

**Data quality now**
- 6,282 rows x 143 columns, no missing values.
- Timestamps are increasing and unique.
- 32 constant columns and 111 variable columns.


In [11]:
def clean_siemens(raw_data):
    """Clean the raw Siemens PLC data and return a new DataFrame."""
    data = raw_data.copy()

    # 1. Remove columns that are empty everywhere.
    data = data.dropna(axis=1, how="all")

    # 2. Remove rows where more than half of the values are missing.
    mostly_missing = data.isna().sum(axis=1) > data.shape[1] / 2
    data = data[~mostly_missing].copy()

    # 3. Convert Boolean columns that are stored as object to bool.
    object_columns = data.select_dtypes(include="object").columns
    if data[object_columns].isna().any().any():
        raise ValueError("Object columns still contain missing values.")
    data[object_columns] = data[object_columns].astype(bool)

    return data

#siemens_clean = clean_siemens(siemens_data)